In [ ]:
!pip install pandas numpy

In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

np.random.seed(42)

# -----------------------------
# Config
# -----------------------------
NUM_INSTANCES = 200
INSTANCE_TYPES = ["m5.large", "c5.large", "t3.medium"]
AVAILABILITY_ZONES = ["us-east-1a", "us-east-1b"]

START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2024, 1, 5)

# -----------------------------
# 1. Launch Logs
# -----------------------------
launch_data = []

for i in range(NUM_INSTANCES):
    launch_time = START_DATE + timedelta(
        minutes=random.randint(0, int((END_DATE - START_DATE).total_seconds() / 60))
    )
    
    launch_data.append({
        "instance_id": f"i-{1000+i}",
        "instance_type": random.choice(INSTANCE_TYPES),
        "availability_zone": random.choice(AVAILABILITY_ZONES),
        "launch_time": launch_time
    })

launch_df = pd.DataFrame(launch_data)

# -----------------------------
# 2. Termination Logs
# -----------------------------
termination_data = []

for _, row in launch_df.iterrows():
    if random.random() < 0.6:  # 60% instances terminate
        
        lifetime_hours = random.randint(1, 24)
        termination_time = row["launch_time"] + timedelta(hours=lifetime_hours)
        
        termination_data.append({
            "instance_id": row["instance_id"],
            "termination_time": termination_time,
            "reason": random.choice(["price-too-high", "capacity"])
        })

termination_df = pd.DataFrame(termination_data)

# -----------------------------
# 3. Spot Price Data (30-min intervals)
# -----------------------------
time_range = pd.date_range(START_DATE, END_DATE, freq="30min")

price_data = []

BASE_PRICES = {
    "m5.large": 0.10,
    "c5.large": 0.08,
    "t3.medium": 0.05
}

for ts in time_range:
    for itype in INSTANCE_TYPES:
        for az in AVAILABILITY_ZONES:
            
            base = BASE_PRICES[itype]
            
            # simulate fluctuation + occasional spikes
            noise = np.random.normal(0, 0.01)
            spike = np.random.choice([0, 0.05], p=[0.9, 0.1])
            
            spot_price = max(0.01, base + noise + spike)
            
            price_data.append({
                "timestamp": ts,
                "instance_type": itype,
                "availability_zone": az,
                "spot_price": round(spot_price, 4),
                "on_demand_price": base + 0.05
            })

price_df = pd.DataFrame(price_data)

# -----------------------------
# Save Files
# -----------------------------
launch_df.to_csv("launch_log.csv", index=False)
termination_df.to_csv("termination_log.csv", index=False)
price_df.to_csv("spot_price.csv", index=False)

print("✅ Dummy dataset created!")

✅ Dummy dataset created!


In [4]:
launch_data = pd.read_csv("launch_log.csv")
launch_data.head()

,instance_id,instance_type,availability_zone,launch_time
0,i-1000,m5.large,us-east-1a,2024-01-02 13:08:00
1,i-1001,t3.medium,us-east-1b,2024-01-04 09:37:00
2,i-1002,t3.medium,us-east-1a,2024-01-02 10:14:00
3,i-1003,c5.large,us-east-1b,2024-01-01 17:24:00
4,i-1004,c5.large,us-east-1a,2024-01-03 09:41:00


In [5]:
launch_data['instance_type'].value_counts()

instance_type
t3.medium    73
m5.large     70
c5.large     57
Name: count, dtype: int64